# add

# info abour patients 

In [1]:
import sys 
from pathlib import Path
sys.path.append(str(Path().resolve().parents[1]))

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent   #to go un up the the root
DATA_DIR = PROJECT_ROOT / "data" 



from python.dataset_creation import (unique_subjects, all_subjects, 
                                    X_ex,X_seq, y, X_info, all_active_units, inputs_seq                                       
                                    )



from utilities.model import  CNNMLPModel



import torch
import numpy as np

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

subject=s1, exercise=e1, label=correct, unit=u2, axis=z, valleys=11
subject=s1, exercise=e1, label=fast, unit=u2, axis=z, valleys=11
subject=s1, exercise=e1, label=low_amplitude, unit=u2, axis=z, valleys=11
subject=s1, exercise=e2, label=correct, unit=u4, axis=y, valleys=11
subject=s1, exercise=e2, label=fast, unit=u4, axis=y, valleys=11
subject=s1, exercise=e2, label=low_amplitude, unit=u4, axis=y, valleys=11
subject=s1, exercise=e3, label=correct, unit=u2, axis=z, valleys=10
subject=s1, exercise=e3, label=fast, unit=u2, axis=z, valleys=11
subject=s1, exercise=e3, label=low_amplitude, unit=u2, axis=z, valleys=11
subject=s1, exercise=e4, label=correct, unit=u2, axis=y, valleys=11
subject=s1, exercise=e4, label=fast, unit=u2, axis=y, valleys=11
subject=s1, exercise=e4, label=low_amplitude, unit=u2, axis=y, valleys=11
subject=s1, exercise=e5, label=correct, unit=u2, axis=x, valleys=11
subject=s1, exercise=e5, label=fast, unit=u2, axis=x, valleys=11
subject=s1, exercise=e5, label=low_ampl

In [2]:
print(type(all_subjects))
print(np.array(all_subjects).shape)
print(all_subjects[:10] if hasattr(all_subjects, "__len__") else all_subjects)

<class 'numpy.ndarray'>
(1193,)
['s1' 's1' 's1' 's1' 's1' 's1' 's1' 's1' 's1' 's1']


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [4]:
def normalize_unit_name(unit):
    unit = str(unit)

    if unit.startswith("u"):
        return unit

    if unit.endswith(".0"):
        unit = unit[:-2]

    return f"u{unit}"


def make_most_active_sensor_version(X_seq, active_units, inputs_seq):
    """
    X_seq shape: (N, features, time)

    Keeps only the channels belonging to the most active unit.
    Example: u2 keeps acc_mag_u2, gyr_mag_u2, mag_mag_u2.
    """
    X_masked = torch.zeros_like(X_seq)

    for i, unit in enumerate(active_units):
        unit = normalize_unit_name(unit)

        keep_indices = [
            j for j, col in enumerate(inputs_seq)
            if col.endswith(f"_{unit}")
        ]

        if len(keep_indices) == 0:
            raise ValueError(
                f"No columns found for active unit {unit}. "
                f"Check most_active_unit values and inputs_seq."
            )

        X_masked[i, keep_indices, :] = X_seq[i, keep_indices, :]

    return X_masked



def normalize_seq_train_val(X_train, X_val):
    """
    X_train, X_val shape: (N, channels, time)
    Normalizes using training-set statistics only.
    """
    mean = X_train.mean(dim=(0, 2), keepdim=True)
    std = X_train.std(dim=(0, 2), keepdim=True) + 1e-6

    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std

    return X_train_norm, X_val_norm


def normalize_info_train_val(X_info_train, X_info_val):
    """
    X_info columns assumed:
        0 = gender
        1 = age
        2 = weight
        3 = height
        4 = original_rep_length

    Normalizes only numeric continuous columns.
    Leaves gender unchanged.
    """
    X_info_train = X_info_train.clone()
    X_info_val = X_info_val.clone()

    numeric_cols = [1, 2, 3, 4]

    mean = X_info_train[:, numeric_cols].mean(dim=0, keepdim=True)
    std = X_info_train[:, numeric_cols].std(dim=0, keepdim=True) + 1e-6

    X_info_train[:, numeric_cols] = (
        X_info_train[:, numeric_cols] - mean
    ) / std

    X_info_val[:, numeric_cols] = (
        X_info_val[:, numeric_cols] - mean
    ) / std

    return X_info_train, X_info_val

# cross validation

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import numpy as np
import torch



fold_results = []

for val_subject in unique_subjects:
    print(f"\n========== Fold: leaving out {val_subject} ==========")

    train_indices = [i for i, s in enumerate(all_subjects) if s != val_subject]
    val_indices = [i for i, s in enumerate(all_subjects) if s == val_subject]

    X_seq_train = X_seq[train_indices]
    X_seq_val = X_seq[val_indices]

    X_ex_train = X_ex[train_indices]
    X_ex_val = X_ex[val_indices]

    X_info_train = X_info[train_indices]
    X_info_val = X_info[val_indices]

    y_train = y[train_indices]
    y_val = y[val_indices]

    train_active_units = all_active_units[train_indices]

    # 1. Normalize using training-fold statistics only
    X_seq_train, X_seq_val = normalize_seq_train_val(
        X_seq_train,
        X_seq_val
    )

    X_info_train, X_info_val = normalize_info_train_val(
        X_info_train,
        X_info_val
    )

    # 2. Create most-active-sensor-only version of TRAINING data only
    X_seq_train_masked = make_most_active_sensor_version(
        X_seq_train,
        train_active_units,
        inputs_seq
    )

    # 3. Double the training set
    X_seq_train_aug = torch.cat(
        [X_seq_train, X_seq_train_masked],
        dim=0
    )

    X_ex_train_aug = torch.cat(
        [X_ex_train, X_ex_train],
        dim=0
    )

    X_info_train_aug = torch.cat(
        [X_info_train, X_info_train],
        dim=0
    )

    y_train_aug = torch.cat(
        [y_train, y_train],
        dim=0
    )

    # 4. Dataset objects
    train_dataset = TensorDataset(
        X_seq_train_aug,
        X_ex_train_aug,
        X_info_train_aug,
        y_train_aug
    )

    val_dataset = TensorDataset(
        X_seq_val,
        X_ex_val,
        X_info_val,
        y_val
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        drop_last=False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False
    )
    # Fresh model for each fold
    model = CNNMLPModel(
        input_dim_seq=X_seq.shape[1],
        input_dim_exercise=X_ex.shape[1],
        input_dim_info=X_info.shape[1],
        num_classes=3,
        use_seq=True,
        use_ex=True,
        use_info=True,
        
    ).to(device)

    criterion = torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=5e-5,
    weight_decay=1e-3
    )

    best_val_loss = float("inf")
    best_preds = None
    best_labels = None

    for epoch in range(70):

        # -------------------------
        # TRAIN
        # -------------------------
        model.train()
        running_loss = 0.0

        for x_seq_b, x_ex_b, x_info_b, y_b in train_loader:
            x_seq_b = x_seq_b.to(device)
            x_ex_b = x_ex_b.to(device)
            x_info_b = x_info_b.to(device)
            y_b = y_b.to(device)

            optimizer.zero_grad()

            outputs = model(x_seq_b, x_ex_b, x_info_b)
            loss = criterion(outputs, y_b)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x_seq_b.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        # -------------------------
        # VALIDATION
        # -------------------------
        model.eval()
        val_running_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for x_seq_b, x_ex_b, x_info_b, y_b in val_loader:
                x_seq_b = x_seq_b.to(device)
                x_ex_b = x_ex_b.to(device)
                x_info_b = x_info_b.to(device)
                y_b = y_b.to(device)

                outputs = model(x_seq_b, x_ex_b, x_info_b)
                loss = criterion(outputs, y_b)

                val_running_loss += loss.item() * x_seq_b.size(0)

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_b.cpu().numpy())

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = accuracy_score(all_labels, all_preds)

        print(
            f"Fold {val_subject} | Epoch {epoch + 1:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_preds = np.array(all_preds)
            best_labels = np.array(all_labels)

    fold_acc = accuracy_score(best_labels, best_preds)
    fold_f1 = f1_score(best_labels, best_preds, average="macro")
    fold_cm = confusion_matrix(best_labels, best_preds)

    fold_results.append({
        "subject": val_subject,
        "val_loss": best_val_loss,
        "accuracy": fold_acc,
        "macro_f1": fold_f1,
        "confusion_matrix": fold_cm
    })

    


========== Fold: leaving out s1 ==========
Fold s1 | Epoch 01 | Train Loss: 1.0592 | Val Loss: 1.0665 | Val Acc: 0.4202
Fold s1 | Epoch 02 | Train Loss: 0.9634 | Val Loss: 1.0537 | Val Acc: 0.4412
Fold s1 | Epoch 03 | Train Loss: 0.8590 | Val Loss: 1.0520 | Val Acc: 0.4580
Fold s1 | Epoch 04 | Train Loss: 0.7531 | Val Loss: 1.0114 | Val Acc: 0.5000
Fold s1 | Epoch 05 | Train Loss: 0.6725 | Val Loss: 1.0161 | Val Acc: 0.5000
Fold s1 | Epoch 06 | Train Loss: 0.6170 | Val Loss: 0.9942 | Val Acc: 0.5084
Fold s1 | Epoch 07 | Train Loss: 0.5709 | Val Loss: 0.9928 | Val Acc: 0.5168
Fold s1 | Epoch 08 | Train Loss: 0.5331 | Val Loss: 1.0420 | Val Acc: 0.5294
Fold s1 | Epoch 09 | Train Loss: 0.5037 | Val Loss: 1.1005 | Val Acc: 0.4958
Fold s1 | Epoch 10 | Train Loss: 0.4780 | Val Loss: 1.0992 | Val Acc: 0.5084
Fold s1 | Epoch 11 | Train Loss: 0.4504 | Val Loss: 1.1598 | Val Acc: 0.4916
Fold s1 | Epoch 12 | Train Loss: 0.4248 | Val Loss: 1.1985 | Val Acc: 0.4958
Fold s1 | Epoch 13 | Train Loss:

KeyboardInterrupt: 

In [ ]:
for r in fold_results:
    print(f"Subject: {r['subject']}")
    print(f"Val Loss: {r['val_loss']:.4f}")
    print(f"Accuracy: {r['accuracy']:.4f}")
    print(f"Macro F1: {r['macro_f1']:.4f}")
    print("Confusion Matrix:")
    print(r["confusion_matrix"])
    print("-" * 40)

Subject: s1
Val Loss: 1.0777
Accuracy: 0.3487
Macro F1: 0.3052
Confusion Matrix:
[[39 40  0]
 [40 39  0]
 [36 39  5]]
----------------------------------------
Subject: s2
Val Loss: 0.5133
Accuracy: 0.8333
Macro F1: 0.8325
Confusion Matrix:
[[64 15  1]
 [24 56  0]
 [ 0  0 80]]
----------------------------------------
Subject: s3
Val Loss: 0.5065
Accuracy: 0.8067
Macro F1: 0.8059
Confusion Matrix:
[[52 27  0]
 [16 62  1]
 [ 1  1 78]]
----------------------------------------
Subject: s4
Val Loss: 0.9280
Accuracy: 0.5210
Macro F1: 0.4491
Confusion Matrix:
[[ 3 77  0]
 [ 0 79  0]
 [ 0 37 42]]
----------------------------------------
Subject: s5
Val Loss: 0.6223
Accuracy: 0.6987
Macro F1: 0.6773
Confusion Matrix:
[[56  1 23]
 [39 31  9]
 [ 0  0 80]]
----------------------------------------


In [ ]:
for s in np.unique(all_subjects):
    mask = np.array(all_subjects) == s
    print(s, np.bincount(y[mask]))

s1 [79 79 80]
s2 [80 80 80]
s3 [79 79 80]
s4 [80 79 79]
s5 [80 79 80]


# model training 


In [5]:
val_subject = "s2"   # subject/patient left out for validation

train_indices = [
    i for i, s in enumerate(all_subjects)
    if s != val_subject
]

val_indices = [
    i for i, s in enumerate(all_subjects)
    if s == val_subject
]

# 0. Split data according to subject
X_seq_train = X_seq[train_indices]
X_seq_val   = X_seq[val_indices]

X_ex_train = X_ex[train_indices]
X_ex_val   = X_ex[val_indices]

X_info_train = X_info[train_indices]
X_info_val   = X_info[val_indices]

y_train = y[train_indices]
y_val   = y[val_indices]

# Active units must also be split, if they are sample-level
train_active_units = [all_active_units[i] for i in train_indices]
val_active_units   = [all_active_units[i] for i in val_indices]

# 1. Normalize using training-fold statistics only
X_seq_train, X_seq_val = normalize_seq_train_val(
    X_seq_train,
    X_seq_val
)

X_info_train, X_info_val = normalize_info_train_val(
    X_info_train,
    X_info_val
)

# Usually X_ex should NOT be normalized if it is one-hot exercise encoding.
# Normalize X_ex only if it contains continuous numerical features.

# 2. Create most-active-sensor-only version of TRAINING data only
X_seq_train_masked = make_most_active_sensor_version(
    X_seq_train,
    train_active_units,
    inputs_seq
)

# 3. Double the training set
X_seq_train_aug = torch.cat(
    [X_seq_train, X_seq_train_masked],
    dim=0
)

X_ex_train_aug = torch.cat(
    [X_ex_train, X_ex_train],
    dim=0
)

X_info_train_aug = torch.cat(
    [X_info_train, X_info_train],
    dim=0
)

y_train_aug = torch.cat(
    [y_train, y_train],
    dim=0
)


train_dataset = TensorDataset(
    X_seq_train_aug,
    X_ex_train_aug,
    X_info_train_aug,
    y_train_aug
)

val_dataset = TensorDataset(
    X_seq_val,
    X_ex_val,
    X_info_val,
    y_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)


model = CNNMLPModel(
    input_dim_seq=X_seq_train.shape[1],
    input_dim_exercise=X_ex_train.shape[1],
    input_dim_info=X_info_train.shape[1],
    num_classes=3,
    use_seq=True,
    use_ex=True,
    use_info=True
).to(device)

In [7]:
import copy

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

best_val_loss = float("inf")
best_preds = None
best_labels = None
best_epoch = None

for epoch in range(20):

    # TRAIN
    model.train()
    running_loss = 0.0

    for x_seq_b, x_ex_b, x_info_b, y_b in train_loader:
        x_seq_b = x_seq_b.to(device)
        x_ex_b = x_ex_b.to(device)
        x_info_b = x_info_b.to(device)
        y_b = y_b.to(device)

       

        optimizer.zero_grad()

        outputs = model(x_seq_b, x_ex_b, x_info_b)
        loss = criterion(outputs, y_b)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x_seq_b.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # VALIDATION
    model.eval()
    val_running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_seq_b, x_ex_b, x_info_b, y_b in val_loader:
            x_seq_b = x_seq_b.to(device)
            x_ex_b = x_ex_b.to(device)
            x_info_b = x_info_b.to(device)
            y_b = y_b.to(device)

            outputs = model(x_seq_b, x_ex_b, x_info_b)
            loss = criterion(outputs, y_b)


            val_running_loss += loss.item() * x_seq_b.size(0)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_b.cpu().numpy())

    val_loss = val_running_loss / len(val_loader.dataset)
    val_acc = accuracy_score(all_labels, all_preds)

    print(
        f"Fold {val_subject} | Epoch {epoch + 1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    # keep best epoch for this fold
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_preds = np.array(all_preds)
        best_labels = np.array(all_labels)
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1

# fold-level metrics
fold_acc = accuracy_score(best_labels, best_preds)
fold_f1 = f1_score(best_labels, best_preds, average="macro")
fold_cm = confusion_matrix(best_labels, best_preds)

results = {}

results[val_subject] = {
    "subject": val_subject,
    "best_epoch": best_epoch,
    "val_loss": best_val_loss,
    "accuracy": fold_acc,
    "macro_f1": fold_f1,
    "confusion_matrix": fold_cm,
    "model_state_dict": best_model_state,
    "predictions": best_preds,
    "labels": best_labels
}

Fold s2 | Epoch 1 | Train Loss: 0.9409 | Val Loss: 0.8166 | Val Acc: 0.8000
Fold s2 | Epoch 2 | Train Loss: 0.7979 | Val Loss: 0.6899 | Val Acc: 0.8292
Fold s2 | Epoch 3 | Train Loss: 0.6628 | Val Loss: 0.5819 | Val Acc: 0.8583
Fold s2 | Epoch 4 | Train Loss: 0.5374 | Val Loss: 0.5394 | Val Acc: 0.8208
Fold s2 | Epoch 5 | Train Loss: 0.4405 | Val Loss: 0.5760 | Val Acc: 0.7792
Fold s2 | Epoch 6 | Train Loss: 0.3503 | Val Loss: 0.5251 | Val Acc: 0.8208
Fold s2 | Epoch 7 | Train Loss: 0.3052 | Val Loss: 0.3965 | Val Acc: 0.8667
Fold s2 | Epoch 8 | Train Loss: 0.2548 | Val Loss: 0.4353 | Val Acc: 0.8333
Fold s2 | Epoch 9 | Train Loss: 0.2089 | Val Loss: 0.4164 | Val Acc: 0.8542
Fold s2 | Epoch 10 | Train Loss: 0.1924 | Val Loss: 0.5477 | Val Acc: 0.8292
Fold s2 | Epoch 11 | Train Loss: 0.1777 | Val Loss: 0.5054 | Val Acc: 0.8083
Fold s2 | Epoch 12 | Train Loss: 0.1564 | Val Loss: 0.4430 | Val Acc: 0.8417
Fold s2 | Epoch 13 | Train Loss: 0.1418 | Val Loss: 0.3735 | Val Acc: 0.8792
Fold s2 

In [8]:
torch.save(results, "fold_results.pth")